13	Ikhlasul Akmal Hanif 	FastPitch	Java 5

In [1]:
import triton
import librosa
import soundfile as sf
from pathlib import Path

# Filter Dataset

In [2]:
import os
import shutil

# Define source and destination paths
jvm_path = "pbl_dataset/jv_id_male/wavs"
jvf_path = "pbl_dataset/jv_id_female/wavs"
audio_path = "pbl_dataset/audio"

# Define the speaker IDs for jvm and jvf
jvm_speaker_ids = {"05970", "05522", "06383"}
jvf_speaker_ids = {"08305", "08002", "04982"}

# Function to move files based on speaker IDs
def move_files(source_path, speaker_ids, destination_path, prefix):
    # Check if the source and destination directories exist
    if not os.path.exists(source_path):
        print(f"Source directory {source_path} does not exist.")
        return
    if not os.path.exists(destination_path):
        os.makedirs(destination_path)
        print(f"Created destination directory {destination_path}.")

    # Loop through each file in the source directory
    for filename in os.listdir(source_path):
        # Check if the file starts with the correct prefix (e.g., "jvm" or "jvf")
        if filename.startswith(prefix):
            # Split the filename to extract the speaker ID after the prefix
            parts = filename.split("_")
            if len(parts) > 1:
                speaker_id = parts[1]  # Extract speaker ID part
                
                # Debugging: Print filename and extracted speaker ID
                print(f"Checking file: {filename}, extracted speaker ID: {speaker_id}")

                # Check if the extracted speaker ID is in the specified speaker ID list
                if speaker_id in speaker_ids:
                    source_file = os.path.join(source_path, filename)
                    destination_file = os.path.join(destination_path, filename)
                    try:
                        # Move the file to the destination directory
                        shutil.move(source_file, destination_file)
                        print(f"Moved {filename} to {destination_path}")
                    except Exception as e:
                        print(f"Failed to move {filename}: {e}")
                else:
                    print(f"Speaker ID {speaker_id} not in target list; skipping file.")
            else:
                print(f"File {filename} does not contain a speaker ID; skipping.")

# Move jvm and jvf files to the audio folder, matching only specific speaker IDs
move_files(jvm_path, jvm_speaker_ids, audio_path, "jvm")
move_files(jvf_path, jvf_speaker_ids, audio_path, "jvf")

print("File transfer complete.")

Created destination directory pbl_dataset/audio.
Checking file: jvm_08178_00387812886.wav, extracted speaker ID: 08178
Speaker ID 08178 not in target list; skipping file.
Checking file: jvm_03314_01333135851.wav, extracted speaker ID: 03314
Speaker ID 03314 not in target list; skipping file.
Checking file: jvm_03727_02009152049.wav, extracted speaker ID: 03727
Speaker ID 03727 not in target list; skipping file.
Checking file: jvm_05219_00327030652.wav, extracted speaker ID: 05219
Speaker ID 05219 not in target list; skipping file.
Checking file: jvm_04285_01284573765.wav, extracted speaker ID: 04285
Speaker ID 04285 not in target list; skipping file.
Checking file: jvm_04285_00319346752.wav, extracted speaker ID: 04285
Speaker ID 04285 not in target list; skipping file.
Checking file: jvm_00027_00910534513.wav, extracted speaker ID: 00027
Speaker ID 00027 not in target list; skipping file.
Checking file: jvm_09724_02014865573.wav, extracted speaker ID: 09724
Speaker ID 09724 not in tar

# Resample

In [3]:
input_dir = Path("pbl_dataset/audio")
output_dir = Path("pbl_dataset/audio_resampled")
output_dir.mkdir(exist_ok=True)

In [4]:
for file_path in input_dir.glob("*.wav"):
    audio, sr = librosa.load(file_path, sr=None)
    audio_resampled = librosa.resample(audio, orig_sr=sr, target_sr=22050)
    output_path = output_dir / file_path.name
    sf.write(output_path, audio_resampled, 22050)
    print(f"Resampled {file_path} to {output_path}")

Resampled pbl_dataset/audio/jvf_04982_00118897617.wav to pbl_dataset/audio_resampled/jvf_04982_00118897617.wav
Resampled pbl_dataset/audio/jvm_06383_01108369204.wav to pbl_dataset/audio_resampled/jvm_06383_01108369204.wav
Resampled pbl_dataset/audio/jvm_06383_01205676079.wav to pbl_dataset/audio_resampled/jvm_06383_01205676079.wav
Resampled pbl_dataset/audio/jvf_04982_01163681269.wav to pbl_dataset/audio_resampled/jvf_04982_01163681269.wav
Resampled pbl_dataset/audio/jvf_08305_00350015030.wav to pbl_dataset/audio_resampled/jvf_08305_00350015030.wav
Resampled pbl_dataset/audio/jvm_05970_00127043277.wav to pbl_dataset/audio_resampled/jvm_05970_00127043277.wav
Resampled pbl_dataset/audio/jvf_08002_00080907059.wav to pbl_dataset/audio_resampled/jvf_08002_00080907059.wav
Resampled pbl_dataset/audio/jvm_06383_02006614642.wav to pbl_dataset/audio_resampled/jvm_06383_02006614642.wav
Resampled pbl_dataset/audio/jvm_05970_01970313200.wav to pbl_dataset/audio_resampled/jvm_05970_01970313200.wav
R

# Create Dataset

In [11]:
import random
from pathlib import Path

data_dir = Path("pbl_dataset/audio_resampled")
tsv_file = Path("pbl_dataset/line_index.tsv")
filelists_dir = Path("pbl_dataset/filelists")
filelists_dir.mkdir(exist_ok=True)

In [12]:
entries = []
with open(tsv_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            # Extract just the filename without the full path
            filename = f"{parts[0]}.wav"
            transcription = parts[2]
            entries.append(f"{filename}|{transcription}")

In [13]:
random.shuffle(entries)
total = len(entries)
train_split = int(0.8 * total)
val_split = int(0.9 * total)

train_entries = entries[:train_split]
val_entries = entries[train_split:val_split]
test_entries = entries[val_split:]

In [14]:
with open(filelists_dir / "train.txt", 'w', encoding='utf-8') as f:
    for entry in train_entries:
        f.write(entry + "\n")

with open(filelists_dir / "val.txt", 'w', encoding='utf-8') as f:
    for entry in val_entries:
        f.write(entry + "\n")

with open(filelists_dir / "test.txt", 'w', encoding='utf-8') as f:
    for entry in test_entries:
        f.write(entry + "\n")

# Evaluation

## MOS

In [83]:
def calculate_mos(scores):
    """
    Calculate the MOS for a single audio sample.
    
    Args:
    - scores (list of int): List of scores from listeners, each between 1 and 5.
    
    Returns:
    - float: The calculated MOS value.
    """
    return sum(scores) / len(scores) if scores else 0

def evaluate_mos(scores_dict):
    """
    Calculate the MOS across multiple audio samples.
    
    Args:
    - scores_dict (dict): A dictionary where each key is the sample identifier, 
                          and the value is a list of scores from listeners for that sample.
    
    Returns:
    - dict: Dictionary with MOS for each sample and overall MOS.
    """
    mos_scores = {}
    total_score = 0
    total_responses = 0
    
    for sample_id, scores in scores_dict.items():
        mos_score = calculate_mos(scores)
        mos_scores[sample_id] = mos_score
        total_score += sum(scores)
        total_responses += len(scores)
        print(f"MOS Score for {sample_id}: {mos_score:.2f}")

    overall_mos = total_score / total_responses if total_responses > 0 else 0
    print(f"\nOverall MOS Score: {overall_mos:.2f}")
    
    return {
        "sample_mos": mos_scores,
        "overall_mos": overall_mos
    }

## MCD

In [91]:
import os
import numpy as np
import librosa

def calculate_mcd13(ref_audio_path, syn_audio_path, sr=22050, n_mfcc=13):
    """
    Calculate MCD13 between reference and synthesized audio files.
    Returns value in dB.
    """
    try:
        ref_audio, _ = librosa.load(ref_audio_path, sr=sr)
        syn_audio, _ = librosa.load(syn_audio_path, sr=sr)
        
        min_len = min(len(ref_audio), len(syn_audio))
        ref_audio = ref_audio[:min_len]
        syn_audio = syn_audio[:min_len]
        
        ref_mfcc = librosa.feature.mfcc(y=ref_audio, sr=sr, n_mfcc=n_mfcc)
        syn_mfcc = librosa.feature.mfcc(y=syn_audio, sr=sr, n_mfcc=n_mfcc)
        
        assert ref_mfcc.shape == syn_mfcc.shape, "MFCC shapes don't match after duration matching"
        
        diff = ref_mfcc - syn_mfcc
        
        sum_squared_diff = np.sum(diff ** 2, axis=0)
        
        mean_squared_diff = np.mean(sum_squared_diff)
        
        mcd = (10 / np.log(10)) * np.sqrt(mean_squared_diff)
        
        return mcd
        
    except Exception as e:
        print(f"Error processing files: {str(e)}")
        return None

def evaluate_mcd13(ref_audio_dir, syn_audio_dir):
    """
    Evaluate MCD13 scores for all matching audio files in the directories.
    """
    mcd13_scores = []
    error_count = 0
    
    for syn_filename in os.listdir(syn_audio_dir):
        if not syn_filename.endswith('.wav'):
            continue
            
        ref_filename = syn_filename
        ref_audio_path = os.path.join(ref_audio_dir, ref_filename)
        syn_audio_path = os.path.join(syn_audio_dir, syn_filename)
        
        if not (os.path.exists(ref_audio_path) and os.path.exists(syn_audio_path)):
            print(f"Missing file for pair: {ref_filename} or {syn_filename}")
            continue
        
        mcd13_score = calculate_mcd13(ref_audio_path, syn_audio_path)
        
        if mcd13_score is not None:
            mcd13_scores.append(mcd13_score)
            print(f"MCD13 Score for {syn_filename}: {mcd13_score:.2f} dB")
        else:
            error_count += 1
    
    if mcd13_scores:
        avg_mcd13 = np.mean(mcd13_scores)
        std_mcd13 = np.std(mcd13_scores)
        print(f"\nResults:")
        print(f"Average MCD13 Score: {avg_mcd13:.2f} dB")
        print(f"Standard Deviation: {std_mcd13:.2f} dB")
        print(f"Number of files processed: {len(mcd13_scores)}")
        if error_count > 0:
            print(f"Number of errors: {error_count}")
    else:
        print("No valid MCD13 scores calculated.")

# Results

In [92]:
# For a single pair of files
ref_audio, _ = librosa.load(ref_audio_dir + '/jvm_05522_00102596192.wav', sr=22050)
syn_audio, _ = librosa.load(syn_audio_dir + '/jvm_05522_00102596192.wav', sr=22050)
print(f"Reference duration: {len(ref_audio)/22050:.2f}s")
print(f"Synthetic duration: {len(syn_audio)/22050:.2f}s")

Reference duration: 2.76s
Synthetic duration: 2.17s


In [101]:
ref_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/jv_audio/wavs/test'
syn_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/output/inference/edu1/audio_output_fp32_fastpitch_waveglow_denoise-0.01/wavs'

evaluate_mcd13(ref_audio_dir, syn_audio_dir)

MCD13 Score for jvm_06383_01205676079.wav: 1203.88 dB
MCD13 Score for jvm_06383_02006614642.wav: 1106.33 dB
MCD13 Score for jvm_05970_01214741788.wav: 1308.02 dB
MCD13 Score for jvm_05522_01617800340.wav: 1184.64 dB
MCD13 Score for jvm_05522_00598267223.wav: 1131.34 dB
MCD13 Score for jvm_05970_00120363075.wav: 1459.03 dB
MCD13 Score for jvm_05522_00777254381.wav: 1248.89 dB
MCD13 Score for jvm_05970_01538233776.wav: 1317.24 dB
MCD13 Score for jvm_05522_01666510959.wav: 1404.61 dB
MCD13 Score for jvm_05522_00102596192.wav: 1184.83 dB
MCD13 Score for jvm_05970_00831736112.wav: 1362.59 dB
MCD13 Score for jvm_05522_02078065000.wav: 1333.87 dB
MCD13 Score for jvm_05522_00885176429.wav: 1285.72 dB
MCD13 Score for jvm_05970_01521872313.wav: 1358.56 dB
MCD13 Score for jvm_06383_00137844423.wav: 1172.43 dB
MCD13 Score for jvm_05522_01657752334.wav: 1132.57 dB
MCD13 Score for jvm_05522_01986797035.wav: 1143.39 dB
MCD13 Score for jvm_06383_01186735094.wav: 1412.90 dB
MCD13 Score for jvm_05970_01

In [57]:
# edu1: 200 epochs, 0.05 LR
ref_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/jv_audio/wavs/test'
syn_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/output/inference/edu1/audio_output_fp32_fastpitch_waveglow_denoise-0.01/wavs'

evaluate_mcd13(ref_audio_dir, syn_audio_dir)

MCD13 Score for jvm_06383_01205676079.wav: 1202.38 dB
MCD13 Score for jvm_06383_02006614642.wav: 1109.33 dB
MCD13 Score for jvm_05970_01214741788.wav: 1308.11 dB
MCD13 Score for jvm_05522_01617800340.wav: 1186.23 dB
MCD13 Score for jvm_05522_00598267223.wav: 1131.78 dB
MCD13 Score for jvm_05970_00120363075.wav: 1410.76 dB
MCD13 Score for jvm_05522_00777254381.wav: 1223.03 dB
MCD13 Score for jvm_05970_01538233776.wav: 1317.31 dB
MCD13 Score for jvm_05522_01666510959.wav: 1399.55 dB
MCD13 Score for jvm_05522_00102596192.wav: 1185.09 dB
MCD13 Score for jvm_05970_00831736112.wav: 1363.65 dB
MCD13 Score for jvm_05522_02078065000.wav: 1331.56 dB
MCD13 Score for jvm_05522_00885176429.wav: 1284.54 dB
MCD13 Score for jvm_05970_01521872313.wav: 1358.65 dB
MCD13 Score for jvm_06383_00137844423.wav: 1172.60 dB
MCD13 Score for jvm_05522_01657752334.wav: 1132.56 dB
MCD13 Score for jvm_05522_01986797035.wav: 1143.70 dB
MCD13 Score for jvm_06383_01186735094.wav: 1414.75 dB
MCD13 Score for jvm_05970_01

In [61]:
# hanif_1: 1000 epochs, 0.5 LR
ref_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/jv_audio/wavs/test'
syn_audio_dir = 'pbl/DeepLearningExamples/PyTorch/SpeechSynthesis/FastPitch/output/inference/standard2/audio_output_fp32_fastpitch_waveglow_denoise-0.01/wavs'

evaluate_mcd13(ref_audio_dir, syn_audio_dir)


Processing jvm_06383_01205676079.wav...

Debug info for jvm_06383_01205676079.wav:
Reference MFCC shape: (13, 249)
Synthetic MFCC shape: (13, 113)
Reference MFCC range: [-555.17, 205.24]
Synthetic MFCC range: [-561.98, 258.25]
Max difference: 364.33
Mean squared difference: 54443.27
Final MCD13 Score: 1013.34 dB

Processing jvm_06383_02006614642.wav...

Debug info for jvm_06383_02006614642.wav:
Reference MFCC shape: (13, 87)
Synthetic MFCC shape: (13, 73)
Reference MFCC range: [-535.51, 254.86]
Synthetic MFCC range: [-495.25, 227.20]
Max difference: 372.01
Mean squared difference: 53889.62
Final MCD13 Score: 1008.18 dB

Processing jvm_05970_01214741788.wav...

Debug info for jvm_05970_01214741788.wav:
Reference MFCC shape: (13, 217)
Synthetic MFCC shape: (13, 140)
Reference MFCC range: [-557.03, 252.18]
Synthetic MFCC range: [-549.87, 242.98]
Max difference: 412.57
Mean squared difference: 40092.77
Final MCD13 Score: 869.60 dB

Processing jvm_05522_01617800340.wav...

Debug info for j

In [30]:
sample_scores = {
    "sample1": [4, 5, 3],
    "sample2": [3, 3, 4],
    "sample3": [5, 4, 5],
    "sample4": [2, 3, 3],
    "sample5": [4, 4, 5],
    "sample6": [3, 4, 3],
    "sample7": [5, 4, 4],
    "sample8": [4, 3, 3],
    "sample9": [3, 4, 2],
    "sample10": [4, 5, 4],
    "sample11": [5, 3, 4],
    "sample12": [3, 2, 3],
    "sample13": [4, 4, 4],
    "sample14": [5, 5, 5],
    "sample15": [2, 3, 2],
    "sample16": [3, 4, 3],
    "sample17": [5, 5, 4],
    "sample18": [3, 4, 5],
    "sample19": [4, 5, 4],
    "sample20": [3, 3, 4]
}

evaluate_mos(sample_scores)

MOS Score for sample1: 4.00
MOS Score for sample2: 3.33
MOS Score for sample3: 4.67
MOS Score for sample4: 2.67
MOS Score for sample5: 4.33
MOS Score for sample6: 3.33
MOS Score for sample7: 4.33
MOS Score for sample8: 3.33
MOS Score for sample9: 3.00
MOS Score for sample10: 4.33
MOS Score for sample11: 4.00
MOS Score for sample12: 2.67
MOS Score for sample13: 4.00
MOS Score for sample14: 5.00
MOS Score for sample15: 2.33
MOS Score for sample16: 3.33
MOS Score for sample17: 4.67
MOS Score for sample18: 4.00
MOS Score for sample19: 4.33
MOS Score for sample20: 3.33

Overall MOS Score: 3.75


{'sample_mos': {'sample1': 4.0,
  'sample2': 3.3333333333333335,
  'sample3': 4.666666666666667,
  'sample4': 2.6666666666666665,
  'sample5': 4.333333333333333,
  'sample6': 3.3333333333333335,
  'sample7': 4.333333333333333,
  'sample8': 3.3333333333333335,
  'sample9': 3.0,
  'sample10': 4.333333333333333,
  'sample11': 4.0,
  'sample12': 2.6666666666666665,
  'sample13': 4.0,
  'sample14': 5.0,
  'sample15': 2.3333333333333335,
  'sample16': 3.3333333333333335,
  'sample17': 4.666666666666667,
  'sample18': 4.0,
  'sample19': 4.333333333333333,
  'sample20': 3.3333333333333335},
 'overall_mos': 3.75}